# 段階 C2：正規化しない合図 U_min を、学習済みの EDM（VP と VE の網）で試す

## 使い方
1. **Colab で開く**：このファイルを Google ドライブに置き、右クリック →「アプリで開く」→ Google Colaboratory。
2. **GPU を選ぶ**：「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」。
3. **ドライブをつなぐ**：② のセルで許可します。C1 の結果は `マイドライブ/C1_results/` にあるものを**読むだけ**で、書き換えません。C2 の結果は `マイドライブ/C2_results/` に書きます。
4. **走らせる順番**：上から順に実行します。① の設定で、何を走らせるかを選べます。
   - `RUN_VP`：VP の網で U_min（2.38、7.35）。M0 と WV の格子は C1 の結果を使い、走らせ直しません。
   - `RUN_VE_BASE`：VE の網で M0・WV_tele の格子・別の種の M0（規則に関わらない部分）。
   - `RUN_VE_RULES`：VE の網で U_min（2.38、7.35）と S_gap（C1 の形。最初に C1 と同じパイロットで枝を決めます）。
   - 最後の ⑩ のセルが、そろった所まで集計・判定・図・zip を作ります（足りない所は「未判定」と出ます）。
5. **途中で切れたときの再開の仕方**：1 束（1,000 枚）ごとにドライブへ書き出しています。切れたら、**もう一度上から順に実行するだけ**で、済んだ束・済んだ特徴量は飛ばして続きから計算します。
6. **持ち帰るもの**：`マイドライブ/C2_results/C2_report_part01.zip`（〜）。数値（json）・図（png）・ログが入っています（画像そのものは入れていません）。

## 時間の見込み（C1 の実測から：VP の M0 1 束 約 232 秒、10 束で約 39 分）
網の評価の量（M0 = 1）に比例するとして見積もります（C1 の実測でも、WV や S_gap の時間は費用にほぼ比例しました）。U_min の費用は、事前登録（版 2）の見込みから 0.65〜0.81 としました。

| 部分 | 網の評価の量（M0 = 1） | VP と同じ速さなら |
|---|---|---|
| `RUN_VP`（U_min 2.38 と 7.35） | 約 1.5 | 約 1 時間 |
| `RUN_VE_BASE`（M0 ×2、WV_tele 7 本） | 約 6.2 | 約 4 時間 |
| `RUN_VE_RULES`（パイロット、U_min ×2、S_gap） | 約 2.4 | 約 1.6 時間 |
| FID の特徴量（1 方法あたり） | — | 1〜2 分 |

- **VE の網（NCSN++）は VP の網（DDPM++）より重い作りなので、VE の部分はこれより長くなる見込みです（測っていません）。** VE の最初の束が終わると、その実測から残りの時間をログに出します。
- 無料枠では一度に全部は終わらないかもしれません。そのときは、日を分けて上から実行し直してください。

## ライセンス
- **このノートブック、c1core.py、c2core.py**：MIT License（© 2026 Taishi Namba）。
- **EDM のコードと学習済みの重み**（NVIDIA、https://github.com/NVlabs/edm）：**CC BY-NC-SA 4.0**（非商用）。このノートブックには含めず、公式の場所から読み込みます。
- **FID の Inception の重み**、**CIFAR-10 の FID の基準の統計量**（EDM が公開）、**CIFAR-10**（Krizhevsky, 2009、torchvision）：公式の場所から読み込みます。条件はそれぞれの配布元に従います。

## 何をしているか（短く）
- u = |D_G − D_net|² / d（束の 5% の点での中央値、毎回の呼び出しで評価 = T1P5）。
- **U_min**（B21 の定義どおり）：2 回目以降の評価で、u がそれまでの最小値の C_MIN 倍以上になった最初の評価で、全部の点を網に切り替えます（後戻りしない）。切り替える前は、全部の点を D_G で進めます。主の C_MIN = 2.38、副は 7.35。
- **VE の網の u の曲線は、計算が終わるまで見ないでください**（汚れていない試験にするため）。このノートブックは、⑩ の前には u の曲線を表示しません。


In [ ]:
# ① 設定
C1_OUT = '/content/drive/MyDrive/C1_results'   # C1 の結果（読むだけ）
OUT = '/content/drive/MyDrive/C2_results'      # C2 の結果
RUN_VP = True         # VP の網：U_min（2.38、7.35）
RUN_VE_BASE = True    # VE の網：M0、WV_tele の格子、別の種の M0
RUN_VE_RULES = True   # VE の網：U_min（2.38、7.35）、S_gap
NET_CHUNK = 250       # 網に一度に通す枚数（メモリ不足なら 125 に）

EDM_BASE = 'https://nvlabs-fi-cdn.nvidia.com/edm/pretrained/'
VP_URL = EDM_BASE + 'edm-cifar10-32x32-uncond-vp.pkl'
VE_URL = EDM_BASE + 'edm-cifar10-32x32-uncond-ve.pkl'
DETECTOR_URL = 'https://api.ngc.nvidia.com/v2/models/nvidia/research/stylegan3/versions/1/files/metrics/inception-2015-12-05.pkl'
REF_URL = 'https://nvlabs-fi-cdn.nvidia.com/edm/fid-refs/cifar10-32x32.npz'


In [ ]:
# ② ドライブと GPU
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, time, pickle, gc
import numpy as np
import torch
assert torch.cuda.is_available(), 'GPU がありません。「ランタイム」→「ランタイムのタイプを変更」で T4 GPU を選んでください。'
dev = torch.device('cuda')
GPU_NAME = torch.cuda.get_device_name(0)
os.makedirs(OUT, exist_ok=True)
print('GPU:', GPU_NAME, '  torch:', torch.__version__)
print('C1 の結果:', C1_OUT, '（ある）' if os.path.isdir(C1_OUT) else '（見つかりません）', '  C2 の書き出し先:', OUT)


In [ ]:
# ③ EDM のコード（公式のリポジトリから。私たちのリポジトリには入れない）
!test -d /content/edm || git clone -q https://github.com/NVlabs/edm.git /content/edm
!cd /content/edm && git rev-parse HEAD > /content/edm_commit.txt
sys.path.insert(0, '/content/edm')
import dnnlib
EDM_COMMIT = open('/content/edm_commit.txt').read().strip()
print('EDM commit:', EDM_COMMIT)


In [ ]:
%%writefile /content/c1core.py
# c1core.py -- 実験場C 段階C1 の中身（MIT License, (c) 2026 Taishi Namba）
#
# 霧が晴れる時刻の合図 S_gap を、学習済みの拡散モデル（EDM の形：D(x, sigma) を返す網）で試すための道具。
# 同じコードを、Colab（torch, GPU）と作業場の CPU 確かめ（numpy）の両方で使う。
#   - 状態 x は (n, d) の配列（torch.Tensor でも numpy.ndarray でもよい）
#   - 網は net_fn(x, sigma) -> D（x と同じ型・形）。sigma は python の float。
# 他者のコード・重み（EDM, Inception）はここには入っていない。ノートブックが公式の場所から読む。

import json
import math
import os
import time

import numpy as np

C1_VERSION = "C1-1.0"

# ---------------------------------------------------------------------------
# 事前登録の定数（C1_事前登録.md の JSON と同じでなければならない。検証で照合する）
# ---------------------------------------------------------------------------
PREREG = {
    # 規則（実験場 B から変えない）
    "factor": 4.0,              # 基準値の 4 倍を超えたら切り替え
    "fog_sigma": 10.0,          # 基準値を作る霧の濃い所（sigma >= 10）
    "subset_frac": 0.05,        # 合図は束の 5% の点だけで作る
    # σ>=10 が霧の濃い所に当たらない場合の代わり（最初の 3 回の呼び出しの中央値）
    "fallback_first_calls": 3,
    # 「σ>=10 が霧の濃い所に当たるか」の判定（パイロット、M0 の軌道、50 歩）
    "pilot_steps": 50,
    "pilot_n": 256,
    "pilot_seedset": 9,
    "min_fog_calls": 3,
    "flat_band_low": [10.0, 20.0],
    "flat_band_high": [40.0, 80.0],
    "flat_ratio_max": 2.0,
    # 本番
    "main_steps": 18,
    "batch": 1000,
    "n_images_main": 10000,
    "seedset_main": 0,
    "seedset_noise": 1,
    "wv_sigma_targets": [40.0, 20.0, 10.0, 5.0, 2.5, 1.25, 0.6],
    # 判定
    "fid_rel_tol": 0.03,        # 画質を落とさない = FID <= 1.03 * FID(M0)
    "hc1_batch_frac": 0.8,      # HC1：束の 8 割以上
    "hc3_cost_max": 0.85,       # HC3：網の評価 <= 0.85 * M0
    "hc4_fid_rel": 0.03,        # HC4：FID <= 1.03 * FID(WV_tele 最良)
    "hc4_cost_margin": 0.10,    # HC4：費用 <= 費用(WV_tele 最良) + 0.10
}

# EDM の標準の目盛り
EDM_SIGMA_MIN = 0.002
EDM_SIGMA_MAX = 80.0
EDM_RHO = 7.0


# ---------------------------------------------------------------------------
# numpy / torch の小さな橋
# ---------------------------------------------------------------------------
def _is_torch(a):
    return type(a).__module__.startswith("torch")


def to_np(a):
    if _is_torch(a):
        return a.detach().cpu().numpy()
    return np.asarray(a)


def _clone(a):
    return a.clone() if _is_torch(a) else a.copy()


def _idx(idx, like):
    if _is_torch(like):
        import torch
        return torch.as_tensor(np.asarray(idx), device=like.device, dtype=torch.long)
    return np.asarray(idx)


def rownorm2(a):
    """各行の二乗ノルム（長さ n の配列）。"""
    return (a * a).reshape(a.shape[0], -1).sum(-1)


# ---------------------------------------------------------------------------
# ガウスのスコア s_G（データの平均と共分散だけ）
# ---------------------------------------------------------------------------
def moments_from_batches(batches):
    """batches: (m, d) の numpy 配列の列。float64 で平均と共分散（1/N で割る）を返す。"""
    n = 0
    s1 = None
    s2 = None
    for b in batches:
        b = np.asarray(b, dtype=np.float64)
        if s1 is None:
            s1 = np.zeros(b.shape[1])
            s2 = np.zeros((b.shape[1], b.shape[1]))
        n += b.shape[0]
        s1 += b.sum(0)
        s2 += b.T @ b
    mu = s1 / n
    cov = s2 / n - np.outer(mu, mu)
    cov = 0.5 * (cov + cov.T)
    return mu, cov, n


def eig_moments(cov):
    lam, U = np.linalg.eigh(np.asarray(cov, dtype=np.float64))
    n_neg = int((lam < 0).sum())
    lam = np.clip(lam, 0.0, None)
    return lam, U, n_neg


class GaussianModel:
    """D_G(x, s) = mu + Sigma (Sigma + s^2 I)^{-1} (x - mu)、固有値分解で計算する。

    mu, lam, U は「状態と同じ型（numpy か torch）」で持つ。to_backend で変換する。
    """

    def __init__(self, mu, lam, U, to_backend=None):
        f = to_backend if to_backend is not None else (lambda a: np.asarray(a, dtype=np.float64))
        self.mu = f(np.asarray(mu, dtype=np.float64))
        self.lam = f(np.asarray(lam, dtype=np.float64))
        self.U = f(np.asarray(U, dtype=np.float64))
        self.d = int(np.asarray(mu).shape[0])

    @classmethod
    def from_cov(cls, mu, cov, to_backend=None):
        lam, U, _ = eig_moments(cov)
        return cls(mu, lam, U, to_backend)

    def denoise(self, x, sigma):
        c = (x - self.mu) @ self.U
        c = c * (self.lam / (self.lam + float(sigma) ** 2))
        return self.mu + c @ self.U.T

    def score(self, x, sigma):
        return (self.denoise(x, sigma) - x) / float(sigma) ** 2

    def teleport(self, x, s_from, s_to):
        """ガウスの確率流 ODE を s_from から s_to まで厳密に解く（Wang & Vastola の analytical teleportation）。"""
        c = (x - self.mu) @ self.U
        c = c * ((self.lam + float(s_to) ** 2) / (self.lam + float(s_from) ** 2)) ** 0.5
        return self.mu + c @ self.U.T


# ---------------------------------------------------------------------------
# EDM の目盛りと Heun 型の決定論的サンプラー（Karras et al. 2022, Alg. 2, S_churn = 0）
# ---------------------------------------------------------------------------
def edm_sigmas(num_steps, sigma_min=EDM_SIGMA_MIN, sigma_max=EDM_SIGMA_MAX, rho=EDM_RHO):
    i = np.arange(num_steps, dtype=np.float64)
    t = (sigma_max ** (1 / rho) + i / (num_steps - 1) * (sigma_min ** (1 / rho) - sigma_max ** (1 / rho))) ** rho
    return np.append(t, 0.0)


def nfe_m0(num_steps):
    return 2 * num_steps - 1


def call_sigmas(sigmas):
    """Heun の各呼び出しの (歩, sigma) の列。"""
    out = []
    N = len(sigmas) - 1
    for i in range(N):
        out.append((i, float(sigmas[i])))
        if i < N - 1:
            out.append((i, float(sigmas[i + 1])))
    return out


def heun_sample(x0, sigmas, D_call, start_step=0):
    """D_call(x, sigma, step, call_index) -> D。start_step から歩き始める（x0 は sigmas[start_step] の点）。"""
    x = x0
    N = len(sigmas) - 1
    ncall = 0
    for i in range(start_step, N):
        tc = float(sigmas[i])
        tn = float(sigmas[i + 1])
        d = (x - D_call(x, tc, i, ncall)) / tc
        ncall += 1
        xn = x + (tn - tc) * d
        if i < N - 1:
            d2 = (xn - D_call(xn, tn, i, ncall)) / tn
            ncall += 1
            xn = x + (tn - tc) * (0.5 * d + 0.5 * d2)
        x = xn
    return x


class CountingNet:
    """網の呼び出しを点の数で数える包み。"""

    def __init__(self, net_fn):
        self.net_fn = net_fn
        self.points = 0

    def __call__(self, x, sigma):
        self.points += int(x.shape[0])
        return self.net_fn(x, sigma)


# ---------------------------------------------------------------------------
# 合図 S_gap の規則（定数は PREREG）
# ---------------------------------------------------------------------------
class GapRule:
    """各呼び出しの統計量 s（束の点での中央値）を受け取り、切り替えるかを返す。

    mode = "fog"  : sigma >= fog_sigma の呼び出しの s の中央値を基準値にする。
                    sigma < fog_sigma になった最初の呼び出しで基準値を固める。以後、s > factor * 基準値 で切り替え。
    mode = "first": 最初の fallback_first_calls 回の呼び出しの s の中央値を基準値にする。
    切り替えたら後戻りしない。
    """

    def __init__(self, mode="fog", factor=None, fog_sigma=None, n_first=None):
        assert mode in ("fog", "first")
        self.mode = mode
        self.factor = PREREG["factor"] if factor is None else factor
        self.fog_sigma = PREREG["fog_sigma"] if fog_sigma is None else fog_sigma
        self.n_first = PREREG["fallback_first_calls"] if n_first is None else n_first
        self.base_vals = []
        self.baseline = None
        self.switched = False
        self.ncalls = 0

    def in_baseline_phase(self, sigma):
        if self.baseline is not None:
            return False
        if self.mode == "fog":
            return sigma >= self.fog_sigma
        return self.ncalls < self.n_first

    def observe(self, sigma, s):
        """True を返したら、この呼び出しから網に切り替える。"""
        if self.switched:
            self.ncalls += 1
            return True
        if self.in_baseline_phase(sigma):
            self.base_vals.append(float(s))
            self.ncalls += 1
            return False
        if self.baseline is None:
            if len(self.base_vals) == 0:
                raise RuntimeError("基準値を作る呼び出しが 1 つもありません（mode=%s）" % self.mode)
            self.baseline = float(np.median(self.base_vals))
        self.ncalls += 1
        if float(s) > self.factor * self.baseline:
            self.switched = True
            return True
        return False


def gap_stat_points(DG, Dn, x):
    """|s_G - s_net|^2 / |s_net|^2 = |D_G - D_net|^2 / |D_net - x|^2（sigma は消える）。点ごとの値（numpy）。"""
    num = to_np(rownorm2(DG - Dn)).astype(np.float64)
    den = to_np(rownorm2(Dn - x)).astype(np.float64)
    return num / np.maximum(den, 1e-300)


# ---------------------------------------------------------------------------
# 方法
# ---------------------------------------------------------------------------
def method_tag(m):
    if m["method"] in ("M0", "S_gap", "Gall"):
        base = m["method"]
    else:
        base = "%s_k%02d" % (m["method"], m["k"])
    return "%s_N%03d_s%d" % (base, m["steps"], m["seedset"])


def run_one(method, k, x_init, sigmas, net_fn, G, rng=None, branch="fog", rule_kw=None, sync=None):
    """一つの束を走らせる。x_init は sigma_max = sigmas[0] の点（= latents * sigmas[0]）。

    返り値：(x_final, info)。info["cost_units"] は「束全体で網 1 回」を単位にした網の評価の回数。
    """
    n = int(x_init.shape[0])
    net = CountingNet(net_fn)
    N = len(sigmas) - 1
    info = {"method": method, "k": k, "n": n, "steps": N, "branch": None}
    if sync:
        sync()
    t0 = time.time()

    if method == "M0":
        x = heun_sample(x_init, sigmas, lambda x, s, i, c: net(x, s))

    elif method == "Gall":
        x = heun_sample(x_init, sigmas, lambda x, s, i, c: G.denoise(x, s))

    elif method == "WV_step":
        # 最初の k 歩（その歩の補正の呼び出しも含む）を D_G で進み、あとは網
        x = heun_sample(x_init, sigmas, lambda x, s, i, c: G.denoise(x, s) if i < k else net(x, s))
        info["switch_sigma"] = float(sigmas[k])

    elif method == "WV_tele":
        # sigma_max から sigmas[k] までガウスで厳密に飛ばし、そこから網で Heun
        xk = G.teleport(x_init, float(sigmas[0]), float(sigmas[k])) if k > 0 else x_init
        x = heun_sample(xk, sigmas, lambda x, s, i, c: net(x, s), start_step=k)
        info["switch_sigma"] = float(sigmas[k])

    elif method == "S_gap":
        if rng is None:
            rng = np.random.default_rng(0)
        kw = dict(rule_kw or {})
        frac = kw.pop("subset_frac", PREREG["subset_frac"])
        rule = GapRule(mode=branch, **kw)
        m = max(1, int(math.ceil(frac * n)))
        trace = []
        state = {"switch_sigma": None, "switch_call": None}

        def D_call(x, s, i, c):
            if rule.switched:
                rule.observe(s, float("nan"))
                return net(x, s)
            DG = G.denoise(x, s)
            sub = np.sort(rng.choice(n, size=m, replace=False))
            isub = _idx(sub, x)
            xs = x[isub]
            Dn_sub = net(xs, s)
            st = float(np.median(gap_stat_points(DG[isub], Dn_sub, xs)))
            dd = float(np.median(to_np(rownorm2(DG[isub] - Dn_sub)).astype(np.float64))) / G.d
            switch = rule.observe(s, st)
            trace.append([c, float(s), st, dd])  # 呼び出し, σ, 合図の統計量, 中央値 |D_G − D_net|²/d（観察用）
            out = _clone(DG)
            out[isub] = Dn_sub
            if switch:
                rest = np.setdiff1d(np.arange(n), sub)
                if len(rest) > 0:
                    irest = _idx(rest, x)
                    out[irest] = net(x[irest], s)
                state["switch_sigma"] = float(s)
                state["switch_call"] = int(c)
            return out

        x = heun_sample(x_init, sigmas, D_call)
        info.update({
            "branch": branch, "subset_m": m, "trace": trace, "baseline": rule.baseline,
            "switched": rule.switched,
            "switch_sigma": state["switch_sigma"] if rule.switched else 0.0,
            "switch_call": state["switch_call"],
        })
    else:
        raise ValueError(method)

    if sync:
        sync()
    info["seconds"] = time.time() - t0
    info["net_points"] = net.points
    info["cost_units"] = net.points / n
    info["cost_ratio"] = info["cost_units"] / nfe_m0(N)
    return x, info


# ---------------------------------------------------------------------------
# パイロット：M0 の軌道に沿って、各呼び出しで全点の統計量と |D_G - D_net|^2 を測る
# ---------------------------------------------------------------------------
def pilot_curve(x_init, sigmas, net_fn, G):
    rows = []

    def D_call(x, s, i, c):
        Dn = net_fn(x, s)
        DG = G.denoise(x, s)
        st = gap_stat_points(DG, Dn, x)
        dd = to_np(rownorm2(DG - Dn)).astype(np.float64) / G.d
        rows.append({"call": int(c), "step": int(i), "sigma": float(s),
                     "stat_median": float(np.median(st)),
                     "stat_q25": float(np.quantile(st, 0.25)), "stat_q75": float(np.quantile(st, 0.75)),
                     "dD2_per_dim_median": float(np.median(dd)), "dD2_per_dim_mean": float(np.mean(dd))})
        return Dn

    heun_sample(x_init, sigmas, D_call)
    return rows


def decide_branch(rows, P=PREREG):
    """事前登録の規則で、σ>=10 が霧の濃い所に当たるかを決める。"""
    s = np.array([r["sigma"] for r in rows])
    v = np.array([r["stat_median"] for r in rows])
    fog = s >= P["fog_sigma"]
    res = {"n_fog_calls": int(fog.sum()), "sigma_max": float(s.max())}
    lo = (s >= P["flat_band_low"][0]) & (s <= P["flat_band_low"][1])
    hi = (s >= P["flat_band_high"][0]) & (s <= P["flat_band_high"][1])
    res["n_low"] = int(lo.sum())
    res["n_high"] = int(hi.sum())
    ok_range = res["sigma_max"] > P["fog_sigma"] and res["n_fog_calls"] >= P["min_fog_calls"]
    if lo.sum() == 0 or hi.sum() == 0 or not ok_range:
        res.update({"ratio_low_over_high": None, "flat": False, "branch": "first",
                    "reason": "sigma の範囲が足りない"})
        return res
    ratio = float(np.median(v[lo]) / np.median(v[hi]))
    flat = (1.0 / P["flat_ratio_max"]) <= ratio <= P["flat_ratio_max"]
    res.update({"ratio_low_over_high": ratio, "flat": bool(flat),
                "branch": "fog" if flat else "first",
                "reason": "平ら" if flat else "平らでない（基準値を最初の %d 回で作る）" % P["fallback_first_calls"]})
    return res


# ---------------------------------------------------------------------------
# 書き出し（1 束 1 ファイル、途中で切れても再開できる）
# ---------------------------------------------------------------------------
def _atomic_json(path, obj):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1)
    os.replace(tmp, path)


def _atomic_npz(path, **arrs):
    tmp = path + ".tmp.npz"
    np.savez_compressed(tmp, **arrs)
    os.replace(tmp, path)


def to_uint8_images(x, img_shape):
    a = to_np(x).astype(np.float64).reshape((-1,) + tuple(img_shape))
    return np.clip(a * 127.5 + 128.0, 0, 255).astype(np.uint8)


def wv_k_grid(sigmas, targets=None):
    targets = PREREG["wv_sigma_targets"] if targets is None else targets
    t = np.asarray(sigmas[1:-1])  # k = 1..N-1
    ks = set()
    for tg in targets:
        ks.add(int(np.argmin(np.abs(np.log(t) - math.log(tg)))) + 1)
    return sorted(ks)


def main_jobs(steps=None, seedset=None, ks=None, with_noise=True):
    steps = PREREG["main_steps"] if steps is None else steps
    seedset = PREREG["seedset_main"] if seedset is None else seedset
    ks = wv_k_grid(edm_sigmas(steps)) if ks is None else ks
    jobs = [{"method": "M0", "k": 0, "steps": steps, "seedset": seedset},
            {"method": "S_gap", "k": 0, "steps": steps, "seedset": seedset}]
    for k in ks:
        jobs.append({"method": "WV_tele", "k": k, "steps": steps, "seedset": seedset})
    for k in ks:
        jobs.append({"method": "WV_step", "k": k, "steps": steps, "seedset": seedset})
    if with_noise:
        jobs.append({"method": "M0", "k": 0, "steps": steps, "seedset": PREREG["seedset_noise"]})
    return jobs


def subset_rng(seedset, steps, b):
    return np.random.default_rng([20261112, int(seedset), int(steps), int(b)])


def phase_pilot(out, net_fn, G, make_latents, n=None, steps=None, seedset=None, log=print):
    n = PREREG["pilot_n"] if n is None else n
    steps = PREREG["pilot_steps"] if steps is None else steps
    seedset = PREREG["pilot_seedset"] if seedset is None else seedset
    os.makedirs(out, exist_ok=True)
    p = os.path.join(out, "pilot.json")
    if os.path.exists(p):
        log("パイロットは済んでいます：" + p)
        with open(p, encoding="utf-8") as f:
            return json.load(f)["decision"]
    sig = edm_sigmas(steps)
    x0 = make_latents(seedset, 0, n) * float(sig[0])
    rows = pilot_curve(x0, sig, net_fn, G)
    dec = decide_branch(rows)
    _atomic_json(p, {"version": C1_VERSION, "steps": steps, "n": n, "seedset": seedset,
                     "rows": rows, "decision": dec, "prereg": PREREG})
    log("パイロットの判定：%s（%s）" % (dec["branch"], dec["reason"]))
    return dec


def load_branch(out):
    with open(os.path.join(out, "pilot.json"), encoding="utf-8") as f:
        return json.load(f)["decision"]["branch"]


def phase_runs(out, jobs, net_fn, G, make_latents, n_images=None, batch=None, img_shape=(3, 32, 32),
               branch=None, sync=None, log=print):
    n_images = PREREG["n_images_main"] if n_images is None else n_images
    batch = PREREG["batch"] if batch is None else batch
    if branch is None:
        branch = load_branch(out)
    n_batches = int(math.ceil(n_images / batch))
    for job in jobs:
        tag = method_tag(job)
        d = os.path.join(out, "runs", tag)
        os.makedirs(d, exist_ok=True)
        sig = edm_sigmas(job["steps"])
        for b in range(n_batches):
            fi = os.path.join(d, "b%03d.npz" % b)
            fj = os.path.join(d, "b%03d.json" % b)
            if os.path.exists(fi) and os.path.exists(fj):
                continue
            nb = min(batch, n_images - b * batch)
            x0 = make_latents(job["seedset"], b, nb) * float(sig[0])
            x, info = run_one(job["method"], job["k"], x0, sig, net_fn, G,
                              rng=subset_rng(job["seedset"], job["steps"], b), branch=branch, sync=sync)
            info.update({"tag": tag, "batch": b, "seedset": job["seedset"], "version": C1_VERSION})
            _atomic_npz(fi, images=to_uint8_images(x, img_shape))
            _atomic_json(fj, info)
            msg = "%s b%03d  費用 %.3f × M0  %.1f 秒" % (tag, b, info["cost_ratio"], info["seconds"])
            if job["method"] == "S_gap":
                msg += "  切り替え σ = %.4g" % info["switch_sigma"]
            log(msg)
            with open(os.path.join(out, "log.txt"), "a", encoding="utf-8") as f:
                f.write(time.strftime("%Y-%m-%d %H:%M:%S ") + msg + "\n")


# ---------------------------------------------------------------------------
# FID
# ---------------------------------------------------------------------------
def fid_from_stats(mu, sigma, mu_ref, sigma_ref):
    """EDM の fid.py と同じ式。"""
    import scipy.linalg
    m = np.square(mu - mu_ref).sum()
    try:
        s, _ = scipy.linalg.sqrtm(np.dot(sigma, sigma_ref), disp=False)
    except TypeError:  # 新しい scipy には disp がない
        s = scipy.linalg.sqrtm(np.dot(sigma, sigma_ref))
    return float(np.real(m + np.trace(sigma + sigma_ref - s * 2)))


def feat_stats(feats):
    feats = np.asarray(feats, dtype=np.float64)
    return feats.mean(0), np.cov(feats, rowvar=False)


def run_complete(out, tag, n_images=None, batch=None):
    n_images = PREREG["n_images_main"] if n_images is None else n_images
    batch = PREREG["batch"] if batch is None else batch
    d = os.path.join(out, "runs", tag)
    nb = int(math.ceil(n_images / batch))
    return all(os.path.exists(os.path.join(d, "b%03d.npz" % b)) for b in range(nb))


def phase_features(out, tags, feat_fn, n_images=None, batch=None, log=print):
    """feat_fn(uint8 の NCHW numpy) -> (m, F) numpy。方法ごとに feats.npy を書く。"""
    n_images = PREREG["n_images_main"] if n_images is None else n_images
    batch = PREREG["batch"] if batch is None else batch
    nb = int(math.ceil(n_images / batch))
    for tag in tags:
        d = os.path.join(out, "runs", tag)
        fp = os.path.join(d, "feats_%d.npy" % n_images)
        if os.path.exists(fp) or not run_complete(out, tag, n_images, batch):
            continue
        fs = []
        for b in range(nb):
            with np.load(os.path.join(d, "b%03d.npz" % b)) as z:
                fs.append(np.asarray(feat_fn(z["images"]), dtype=np.float64))
        np.save(fp + ".tmp.npy", np.concatenate(fs, 0))
        os.replace(fp + ".tmp.npy", fp)
        log("特徴量：%s" % tag)


# ---------------------------------------------------------------------------
# 集計と判定
# ---------------------------------------------------------------------------
def collect(out, n_images=None, batch=None, ref=None):
    n_images = PREREG["n_images_main"] if n_images is None else n_images
    batch = PREREG["batch"] if batch is None else batch
    nb = int(math.ceil(n_images / batch))
    res = {}
    rd = os.path.join(out, "runs")
    if not os.path.isdir(rd):
        return res
    for tag in sorted(os.listdir(rd)):
        d = os.path.join(rd, tag)
        infos = []
        for b in range(nb):
            fj = os.path.join(d, "b%03d.json" % b)
            if os.path.exists(fj):
                with open(fj, encoding="utf-8") as f:
                    infos.append(json.load(f))
        if not infos:
            continue
        r = {"method": infos[0]["method"], "k": infos[0]["k"], "steps": infos[0]["steps"],
             "seedset": infos[0]["seedset"], "n_batches_done": len(infos), "complete": len(infos) == nb,
             "cost_ratio_mean": float(np.mean([i["cost_ratio"] for i in infos])),
             "seconds_total": float(np.sum([i["seconds"] for i in infos]))}
        if "switch_sigma" in infos[0]:
            sw = [float(i["switch_sigma"]) for i in infos]
            r["switch_sigmas"] = sw
            r["switch_sigma_median"] = float(np.median(sw))
        if r["method"] == "S_gap":
            r["baselines"] = [i["baseline"] for i in infos]
            r["branch"] = infos[0]["branch"]
        fp = os.path.join(d, "feats_%d.npy" % n_images)
        if ref is not None and os.path.exists(fp):
            mu, sg = feat_stats(np.load(fp))
            r["fid"] = fid_from_stats(mu, sg, ref[0], ref[1])
        res[tag] = r
    return res


def judge(res, steps=None, seedset=None, P=PREREG):
    """HC1〜HC4 を事前登録の基準で判定する。足りないものがあれば「未判定」。"""
    steps = P["main_steps"] if steps is None else steps
    seedset = P["seedset_main"] if seedset is None else seedset
    sig = edm_sigmas(steps)
    T = lambda m, k=0: method_tag({"method": m, "k": k, "steps": steps, "seedset": seedset})
    out = {"steps": steps, "seedset": seedset}
    m0 = res.get(T("M0"))
    sg = res.get(T("S_gap"))
    if not m0 or "fid" not in m0 or not sg or "fid" not in sg:
        out["status"] = "未判定（M0 か S_gap の FID がまだありません）"
        return out
    tol = (1 + P["fid_rel_tol"]) * m0["fid"]
    out["fid_M0"] = m0["fid"]
    out["fid_tol"] = tol
    out["fid_S_gap"] = sg["fid"]
    out["cost_S_gap"] = sg["cost_ratio_mean"]
    out["switch_sigmas_S_gap"] = sg["switch_sigmas"]

    def grid(meth):
        g = []
        for tag, r in res.items():
            if r["method"] == meth and r["steps"] == steps and r["seedset"] == seedset and "fid" in r:
                g.append((r["k"], r["fid"], r["cost_ratio_mean"]))
        return sorted(g)

    gs = grid("WV_step")
    gt = grid("WV_tele")
    out["grid_WV_step"] = gs
    out["grid_WV_tele"] = gt

    # HC1：WV_step の格子で、小さい k から続けて「画質を落とさない」最大の k
    kstar = 0
    for k, f, c in gs:
        if f <= tol:
            kstar = k
        else:
            break
    out["k_star_WV_step"] = kstar
    if not gs:
        out["HC1"] = "未判定（WV_step の格子がありません）"
    elif kstar == 0:
        out["sigma_safe"] = None
        out["HC1"] = "不支持（WV_step で画質を落とさず飛ばせる k がない）"
    else:
        ss = float(sig[kstar])
        out["sigma_safe"] = ss
        sw = np.asarray(sg["switch_sigmas"])
        frac = float(np.mean((sw >= ss * (1 - 1e-9)) & (sw > 0)))
        out["HC1_frac"] = frac
        out["HC1"] = "支持" if frac >= P["hc1_batch_frac"] else "不支持"
    # HC2
    out["HC2"] = "支持" if sg["fid"] <= tol else "不支持"
    # HC3
    out["HC3"] = "支持" if sg["cost_ratio_mean"] <= P["hc3_cost_max"] else "不支持"
    # HC4：WV_tele の格子で画質を落とさない k のうち最も安い（最大の k）
    safe_t = [(k, f, c) for (k, f, c) in gt if f <= tol]
    if not gt:
        out["HC4"] = "未判定（WV_tele の格子がありません）"
    else:
        if safe_t:
            kb, fb, cb = max(safe_t, key=lambda z: z[0])
        else:
            kb, fb, cb = 0, m0["fid"], 1.0
        out["WV_tele_best"] = {"k": kb, "fid": fb, "cost": cb, "sigma": float(sig[kb])}
        ok = sg["fid"] <= (1 + P["hc4_fid_rel"]) * fb and sg["cost_ratio_mean"] <= cb + P["hc4_cost_margin"]
        out["HC4"] = "支持" if ok else "不支持"
    noise = res.get(method_tag({"method": "M0", "k": 0, "steps": steps, "seedset": P["seedset_noise"]}))
    if noise and "fid" in noise:
        out["fid_M0_other_seeds"] = noise["fid"]
    out["status"] = "判定済み"
    return out


# ---------------------------------------------------------------------------
# 図
# ---------------------------------------------------------------------------
def make_figures(out, res, judged, steps=None, seedset=None):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    steps = PREREG["main_steps"] if steps is None else steps
    seedset = PREREG["seedset_main"] if seedset is None else seedset
    fd = os.path.join(out, "figures")
    os.makedirs(fd, exist_ok=True)
    files = []
    # 図 1：パイロットの |D_G - D_net|^2 と合図の統計量、切り替えの位置
    pp = os.path.join(out, "pilot.json")
    if os.path.exists(pp):
        with open(pp, encoding="utf-8") as f:
            pil = json.load(f)
        rows = pil["rows"]
        s = np.array([r["sigma"] for r in rows])
        fig, ax = plt.subplots(1, 2, figsize=(11, 4))
        ax[0].loglog(s, [r["dD2_per_dim_median"] for r in rows], "o-", ms=3, label="median |D_G - D_net|^2 / d")
        ax[1].loglog(s, [r["stat_median"] for r in rows], "o-", ms=3, label="S_gap stat (median)")
        sgr = res.get(method_tag({"method": "S_gap", "k": 0, "steps": steps, "seedset": seedset}))
        for a in ax:
            a.axvline(PREREG["fog_sigma"], color="gray", ls=":", label="sigma = 10")
            if sgr:
                for i, sw in enumerate(sgr["switch_sigmas"]):
                    if sw > 0:
                        a.axvline(sw, color="C3", alpha=0.3, label="S_gap switch" if i == 0 else None)
            if judged.get("sigma_safe"):
                a.axvline(judged["sigma_safe"], color="C2", ls="--", label="sigma_safe (WV_step)")
            a.invert_xaxis()
            a.set_xlabel("sigma")
            a.legend(fontsize=7)
        ax[1].set_title("branch: %s" % pil["decision"]["branch"])
        ax[0].set_title("pilot (M0 trajectories, %d steps)" % pil["steps"])
        fig.tight_layout()
        p = os.path.join(fd, "fig1_gaussian_gap_and_switch.png")
        fig.savefig(p, dpi=130)
        plt.close(fig)
        files.append(p)
    # 図 2：FID と費用
    fig, ax = plt.subplots(figsize=(6, 4.5))
    any_pt = False
    for meth, mk in (("WV_tele", "s-"), ("WV_step", "^-")):
        g = sorted([(r["cost_ratio_mean"], r["fid"]) for r in res.values()
                    if r["method"] == meth and r["steps"] == steps and r["seedset"] == seedset and "fid" in r])
        if g:
            ax.plot([a for a, b in g], [b for a, b in g], mk, label=meth)
            any_pt = True
    for meth, mk in (("M0", "ko"), ("S_gap", "r*")):
        r = res.get(method_tag({"method": meth, "k": 0, "steps": steps, "seedset": seedset}))
        if r and "fid" in r:
            ax.plot([r["cost_ratio_mean"]], [r["fid"]], mk, ms=11, label=meth)
            any_pt = True
    if judged.get("fid_tol"):
        ax.axhline(judged["fid_tol"], color="gray", ls=":", label="1.03 x FID(M0)")
    if any_pt:
        ax.set_xlabel("net evaluations / M0")
        ax.set_ylabel("FID (%d images)" % PREREG["n_images_main"])
        ax.legend(fontsize=8)
        fig.tight_layout()
        p = os.path.join(fd, "fig2_fid_vs_cost.png")
        fig.savefig(p, dpi=130)
        files.append(p)
    plt.close(fig)
    # 図 3：S_gap の統計量の軌跡（束ごと）
    d = os.path.join(out, "runs", method_tag({"method": "S_gap", "k": 0, "steps": steps, "seedset": seedset}))
    if os.path.isdir(d):
        fig, ax = plt.subplots(figsize=(6, 4.5))
        for fj in sorted(x for x in os.listdir(d) if x.endswith(".json")):
            with open(os.path.join(d, fj), encoding="utf-8") as f:
                inf = json.load(f)
            tr = np.array(inf["trace"]) if inf["trace"] else np.zeros((0, 3))
            if len(tr):
                ax.loglog(tr[:, 1], tr[:, 2], ".-", alpha=0.5)
            if inf["baseline"]:
                ax.axhline(PREREG["factor"] * inf["baseline"], color="C3", alpha=0.2)
        ax.invert_xaxis()
        ax.set_xlabel("sigma")
        ax.set_ylabel("stat (median over 5% subset)")
        ax.set_title("S_gap traces; red = 4 x baseline")
        fig.tight_layout()
        p = os.path.join(fd, "fig3_sgap_traces.png")
        fig.savefig(p, dpi=130)
        plt.close(fig)
        files.append(p)
    return files


def phase_report(out, ref, steps=None, seedset=None, n_images=None, batch=None, extra=None, log=print):
    res = collect(out, n_images=n_images, batch=batch, ref=ref)
    judged = judge(res, steps=steps, seedset=seedset)
    figs = make_figures(out, res, judged, steps=steps, seedset=seedset)
    rep = os.path.join(out, "report")
    os.makedirs(rep, exist_ok=True)
    summ = {"version": C1_VERSION, "prereg": PREREG, "results": res, "judgement": judged,
            "figures": [os.path.basename(f) for f in figs]}
    if extra:
        summ["extra"] = extra
    _atomic_json(os.path.join(rep, "summary_C1.json"), summ)
    for k in ("HC1", "HC2", "HC3", "HC4"):
        if k in judged:
            log("%s：%s" % (k, judged[k]))
    log("状態：%s" % judged.get("status"))
    return summ


# ---------------------------------------------------------------------------
# 持ち帰る結果を zip にする（画像の npz と特徴量は入れない。1 つ 100 ファイル以下、パスを保つ）
# ---------------------------------------------------------------------------
def pack_report(out, name="C1_report", max_files=100, top="C1_results"):
    import zipfile
    files = []
    for dp, dn, fn in os.walk(out):
        dn.sort()
        for f in sorted(fn):
            if f.endswith((".json", ".png", ".txt")) and ".tmp" not in f:
                files.append(os.path.join(dp, f))
    for old in os.listdir(out):
        if old.startswith(name + "_part") and old.endswith(".zip"):
            os.remove(os.path.join(out, old))
    zips = []
    for i in range(0, len(files), max_files):
        zp = os.path.join(out, "%s_part%02d.zip" % (name, i // max_files + 1))
        with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
            for f in files[i:i + max_files]:
                z.write(f, os.path.join(top, os.path.relpath(f, out)))
        zips.append(zp)
    return zips, len(files)


In [ ]:
%%writefile /content/c2core.py
# c2core.py -- 実験場C 段階C2 の中身（MIT License, (c) 2026 Taishi Namba）
#
# 正規化しない合図 U_min を、学習済みの EDM（VP と VE の網）で試す。
#   u = 束の 5% の点での |D_G − D_net|² / d の中央値（呼び出しごと、時間の間引きなし = T1P5）
#   U_min（B21_rule_for_C.md §3 の定義どおり）：
#          k ≥ 1 で u_k ≥ C_MIN × min(u_0, …, u_{k−1}) となった最初の評価で、全部の点を網に切り替える
#          （1 回目の評価では切り替えない。後戻りしない。切り替える前は、全部の点を D_G で進める）
# c1core.py（C1 で Colab が走らせたものと一字一句同じ）を import し、手を入れない。

import json
import math
import os
import time

import numpy as np

import c1core as C1

C2_VERSION = "C2-1.1"

EDM_BASE = "https://nvlabs-fi-cdn.nvidia.com/edm/pretrained/"

# ---------------------------------------------------------------------------
# 事前登録の定数（C2_事前登録.md の JSON と同じでなければならない。検証で照合する）
# ---------------------------------------------------------------------------
PREREG2 = {
    "c_min_main": 2.38,
    "c_min_sub": 7.35,
    "subset_frac": 0.05,
    "time_thin": 1,
    "steps": 18,
    "batch": 1000,
    "n_images": 10000,
    "seedset_main": 0,
    "seedset_noise": 1,
    "wv_sigma_targets": [40.0, 20.0, 10.0, 5.0, 2.5, 1.25, 0.6],
    "fid_rel_tol": 0.03,
    "cost_max": 0.85,
    "grid_fid_rel": 0.03,
    "grid_cost_margin": 0.10,
    "vp_model": "edm-cifar10-32x32-uncond-vp.pkl",
    "ve_model": "edm-cifar10-32x32-uncond-ve.pkl",
    "sampler_sigma_min": 0.002,
    "sampler_sigma_max": 80.0,
    "sampler_rho": 7.0,
    "replication_rel_tol": 1e-4,
    "rule_inequality": ">=",
    "pre_switch_update": "D_G for all points (B21)",
    "subset_count": "max(1, round(0.05 * n))",
}


# ---------------------------------------------------------------------------
# サンプラーの目盛り（EDM の edm_sampler と同じ決め方）
# ---------------------------------------------------------------------------
def sampler_range(net_sigma_min, net_sigma_max, P=PREREG2):
    """edm_sampler：sigma_min = max(0.002, net.sigma_min)、sigma_max = min(80, net.sigma_max)。"""
    return max(P["sampler_sigma_min"], float(net_sigma_min)), min(P["sampler_sigma_max"], float(net_sigma_max))


def sigmas_for(steps, srange=None, P=PREREG2):
    lo, hi = srange if srange is not None else (P["sampler_sigma_min"], P["sampler_sigma_max"])
    return C1.edm_sigmas(steps, sigma_min=lo, sigma_max=hi, rho=P["sampler_rho"])


# ---------------------------------------------------------------------------
# 規則 U_min
# ---------------------------------------------------------------------------
class UMinRule:
    """observe(u) が True を返したら、その評価から網に切り替える。

    B21_rule_for_C.md §3 の定義どおり（評価の番号 k = 0, 1, 2, …）。
    - k = 0（1 回目の評価）では切り替えない（最小値がまだない）。
    - k ≥ 1：u_k ≥ c_min × min(u_0, …, u_{k−1}) なら切り替え（等しい場合も切り替える）。
      （c_min ≥ 1 では、最小値に今の評価を含めても含めなくても同じ判定になる。）
    - 切り替えたら後戻りしない。
    """

    def __init__(self, c_min):
        self.c_min = float(c_min)
        self.min = None
        self.n_eval = 0
        self.switched = False
        self.min_at_switch = None

    def observe(self, u):
        if self.switched:
            return True
        u = float(u)
        self.n_eval += 1
        if self.min is not None and u >= self.c_min * self.min:
            self.switched = True
            self.min_at_switch = self.min
            return True
        self.min = u if self.min is None else min(self.min, u)
        return False


def replay_umin(us, c_min):
    """u の列に規則を当て、切り替えた評価の番号（0 始まり）を返す。切り替えなければ None。"""
    r = UMinRule(c_min)
    for i, u in enumerate(us):
        if r.observe(u):
            return i
    return None


def u_points(DG, Dn):
    """点ごとの |D_G − D_net|² / d（numpy）。"""
    d = int(np.prod(DG.shape[1:]))
    return C1.to_np(C1.rownorm2(DG - Dn)).astype(np.float64) / d


# ---------------------------------------------------------------------------
# 一つの束を U_min で走らせる（T1P5）
# ---------------------------------------------------------------------------
def run_umin(c_min, x_init, sigmas, net_fn, G, rng, frac=None, sync=None):
    frac = PREREG2["subset_frac"] if frac is None else frac
    n = int(x_init.shape[0])
    net = C1.CountingNet(net_fn)
    N = len(sigmas) - 1
    rule = UMinRule(c_min)
    m = max(1, int(round(frac * n)))  # B21：round(0.05 × 束の大きさ)、少なくとも 1 点
    trace = []
    state = {"switch_sigma": None, "switch_call": None}
    if sync:
        sync()
    t0 = time.time()

    def D_call(x, s, i, c):
        if rule.switched:
            return net(x, s)
        DG = G.denoise(x, s)
        sub = np.sort(rng.choice(n, size=m, replace=False))
        isub = C1._idx(sub, x)
        xs = x[isub]
        Dn_sub = net(xs, s)
        DGs = DG[isub]
        u = float(np.median(u_points(DGs, Dn_sub)))
        gap = float(np.median(C1.gap_stat_points(DGs, Dn_sub, xs)))
        prev_min = rule.min
        switch = rule.observe(u)
        trace.append([int(c), float(s), u, gap, prev_min])
        if not switch:
            # B21：切り替える前は、全部の点を D_G で進める（網を評価した 5% の点も）
            return DG
        out = C1._clone(DG)
        out[isub] = Dn_sub
        rest = np.setdiff1d(np.arange(n), sub)
        if len(rest) > 0:
            irest = C1._idx(rest, x)
            out[irest] = net(x[irest], s)
        state["switch_sigma"] = float(s)
        state["switch_call"] = int(c)
        return out

    x = C1.heun_sample(x_init, sigmas, D_call)
    if sync:
        sync()
    info = {"method": "U_min", "k": 0, "c_min": float(c_min), "n": n, "steps": N, "subset_m": m,
            "trace": trace, "trace_columns": ["call", "sigma", "u", "gap_stat", "min_before"],
            "switched": rule.switched,
            "switch_sigma": state["switch_sigma"] if rule.switched else 0.0,
            "switch_call": state["switch_call"], "min_at_switch": rule.min_at_switch,
            "seconds": time.time() - t0, "net_points": net.points}
    info["cost_units"] = net.points / n
    info["cost_ratio"] = info["cost_units"] / C1.nfe_m0(N)
    return x, info


# ---------------------------------------------------------------------------
# 仕事の名前と一覧
# ---------------------------------------------------------------------------
def tag(job):
    if job["method"] == "U_min":
        return "U_min_c%s_N%03d_s%d" % (("%g" % job["c_min"]), job["steps"], job["seedset"])
    return C1.method_tag(job)


def umin_jobs(cs=None, steps=None, seedset=None):
    cs = [PREREG2["c_min_main"], PREREG2["c_min_sub"]] if cs is None else cs
    steps = PREREG2["steps"] if steps is None else steps
    seedset = PREREG2["seedset_main"] if seedset is None else seedset
    return [{"method": "U_min", "k": 0, "c_min": c, "steps": steps, "seedset": seedset} for c in cs]


def ve_base_jobs(sigmas, steps=None):
    """規則に関わらない仕事（M0、WV_tele の格子、別の種の M0）。安い k から。"""
    steps = PREREG2["steps"] if steps is None else steps
    s0 = PREREG2["seedset_main"]
    ks = C1.wv_k_grid(sigmas, PREREG2["wv_sigma_targets"])
    jobs = [{"method": "M0", "k": 0, "steps": steps, "seedset": s0}]
    jobs += [{"method": "WV_tele", "k": k, "steps": steps, "seedset": s0} for k in sorted(ks, reverse=True)]
    jobs += [{"method": "M0", "k": 0, "steps": steps, "seedset": PREREG2["seedset_noise"]}]
    return jobs


def ve_rule_jobs(steps=None):
    """規則の仕事：主の U_min（2.38）、副の U_min（7.35）、S_gap（C1 の形）。"""
    steps = PREREG2["steps"] if steps is None else steps
    s0 = PREREG2["seedset_main"]
    jobs = umin_jobs(steps=steps)
    jobs.append({"method": "S_gap", "k": 0, "steps": steps, "seedset": s0})
    return jobs


# ---------------------------------------------------------------------------
# 走らせる（1 束 1 ファイル、再開できる）
# ---------------------------------------------------------------------------
def phase_runs2(out, jobs, net_fn, G, make_latents, srange=None, branch=None, n_images=None, batch=None,
                img_shape=(3, 32, 32), sync=None, log=print):
    n_images = PREREG2["n_images"] if n_images is None else n_images
    batch = PREREG2["batch"] if batch is None else batch
    nb = int(math.ceil(n_images / batch))
    for job in jobs:
        tg = tag(job)
        d = os.path.join(out, "runs", tg)
        os.makedirs(d, exist_ok=True)
        sig = sigmas_for(job["steps"], srange)
        for b in range(nb):
            fi = os.path.join(d, "b%03d.npz" % b)
            fj = os.path.join(d, "b%03d.json" % b)
            if os.path.exists(fi) and os.path.exists(fj):
                continue
            n = min(batch, n_images - b * batch)
            x0 = make_latents(job["seedset"], b, n) * float(sig[0])
            rng = C1.subset_rng(job["seedset"], job["steps"], b)
            if job["method"] == "U_min":
                x, info = run_umin(job["c_min"], x0, sig, net_fn, G, rng, sync=sync)
            else:
                if job["method"] == "S_gap" and branch is None:
                    raise RuntimeError("S_gap には枝（パイロットの判定）が要ります")
                x, info = C1.run_one(job["method"], job["k"], x0, sig, net_fn, G, rng=rng,
                                     branch=branch or "fog", sync=sync)
            info.update({"tag": tg, "batch": b, "seedset": job["seedset"], "version": C2_VERSION,
                         "sigma_range": [float(sig[-2]), float(sig[0])]})
            C1._atomic_npz(fi, images=C1.to_uint8_images(x, img_shape))
            C1._atomic_json(fj, info)
            msg = "%s b%03d  費用 %.3f × M0  %.1f 秒" % (tg, b, info["cost_ratio"], info["seconds"])
            if "switch_sigma" in info and job["method"] in ("U_min", "S_gap"):
                msg += "  切り替え σ = %.4g" % info["switch_sigma"]
            log(msg)
            with open(os.path.join(out, "log.txt"), "a", encoding="utf-8") as f:
                f.write(time.strftime("%Y-%m-%d %H:%M:%S ") + msg + "\n")


def phase_pilot2(out, net_fn, G, make_latents, srange=None, log=print):
    """C1 と同じパイロット（50 歩、256 枚、種の組 9）。S_gap の枝を C1 の規則で決める。"""
    P = C1.PREREG
    os.makedirs(out, exist_ok=True)
    p = os.path.join(out, "pilot.json")
    if os.path.exists(p):
        with open(p, encoding="utf-8") as f:
            return json.load(f)["decision"]
    sig = sigmas_for(P["pilot_steps"], srange)
    x0 = make_latents(P["pilot_seedset"], 0, P["pilot_n"]) * float(sig[0])
    rows = C1.pilot_curve(x0, sig, net_fn, G)
    dec = C1.decide_branch(rows)
    C1._atomic_json(p, {"version": C2_VERSION, "steps": P["pilot_steps"], "n": P["pilot_n"],
                        "seedset": P["pilot_seedset"], "rows": rows, "decision": dec,
                        "sigma_range": [float(sig[-2]), float(sig[0])]})
    log("パイロット（S_gap の枝）：%s（%s）" % (dec["branch"], dec["reason"]))
    return dec


# ---------------------------------------------------------------------------
# 集計
# ---------------------------------------------------------------------------
def collect2(out, ref=None, n_images=None, batch=None):
    n_images = PREREG2["n_images"] if n_images is None else n_images
    batch = PREREG2["batch"] if batch is None else batch
    res = C1.collect(out, n_images=n_images, batch=batch, ref=ref)
    for tg, r in res.items():
        if r["method"] == "U_min":
            with open(os.path.join(out, "runs", tg, "b000.json"), encoding="utf-8") as f:
                r["c_min"] = json.load(f)["c_min"]
            r["switched_all"] = r["switch_sigmas"] and all(s > 0 for s in r["switch_sigmas"])
    return res


def judge_rule(res, rule_tag, steps=None, seedset=None, P=PREREG2, srange=None):
    """規則（U_min など）の二つの判定。
    a：FID ≤ 1.03 × FID(M0)、かつ 費用 ≤ 0.85 × M0
    b：WV_tele の格子で「画質を落とさない」最も安い k（なければ M0）と比べ、
       FID ≤ 1.03 × FID(WV_tele(k_best))、かつ 費用 ≤ 費用(k_best) + 0.10
    """
    steps = P["steps"] if steps is None else steps
    seedset = P["seedset_main"] if seedset is None else seedset
    sig = sigmas_for(steps, srange)
    m0 = res.get(C1.method_tag({"method": "M0", "k": 0, "steps": steps, "seedset": seedset}))
    r = res.get(rule_tag)
    out = {"rule": rule_tag}
    if not m0 or "fid" not in m0 or not r or "fid" not in r or not r.get("complete"):
        out["status"] = "未判定（M0 か規則の FID がまだありません）"
        return out
    tol = (1 + P["fid_rel_tol"]) * m0["fid"]
    out.update({"fid_M0": m0["fid"], "fid_tol": tol, "fid_rule": r["fid"], "cost_rule": r["cost_ratio_mean"],
                "switch_sigmas": r.get("switch_sigmas")})
    out["a_fid_ok"] = r["fid"] <= tol
    out["a_cost_ok"] = r["cost_ratio_mean"] <= P["cost_max"]
    out["a"] = "支持" if (out["a_fid_ok"] and out["a_cost_ok"]) else "不支持"
    grid = sorted((v["k"], v["fid"], v["cost_ratio_mean"]) for v in res.values()
                  if v["method"] == "WV_tele" and v["steps"] == steps and v["seedset"] == seedset
                  and "fid" in v and v.get("complete"))
    out["grid_WV_tele"] = grid
    n_grid = len(C1.wv_k_grid(sig, P["wv_sigma_targets"]))
    if len(grid) < n_grid:
        out["b"] = "未判定（WV_tele の格子が %d/%d 本）" % (len(grid), n_grid)
    else:
        safe = [g for g in grid if g[1] <= tol]
        kb, fb, cb = max(safe, key=lambda z: z[0]) if safe else (0, m0["fid"], 1.0)
        out["WV_tele_best"] = {"k": kb, "fid": fb, "cost": cb, "sigma": float(sig[kb])}
        ok = r["fid"] <= (1 + P["grid_fid_rel"]) * fb and r["cost_ratio_mean"] <= cb + P["grid_cost_margin"]
        out["b_fid_ok"] = r["fid"] <= (1 + P["grid_fid_rel"]) * fb
        out["b_cost_ok"] = r["cost_ratio_mean"] <= cb + P["grid_cost_margin"]
        out["b"] = "支持" if ok else "不支持"
    noise = res.get(C1.method_tag({"method": "M0", "k": 0, "steps": steps, "seedset": P["seedset_noise"]}))
    if noise and "fid" in noise:
        out["fid_M0_other_seeds"] = noise["fid"]
        out["fid_seed_spread"] = abs(noise["fid"] - m0["fid"])
        out["rule_minus_M0"] = r["fid"] - m0["fid"]
        out["within_seed_spread"] = abs(r["fid"] - m0["fid"]) <= abs(noise["fid"] - m0["fid"])
    out["status"] = "判定済み"
    return out


def judge_all(res_vp, res_ve, srange_vp=None, srange_ve=None, P=PREREG2):
    main = tag(umin_jobs(cs=[P["c_min_main"]])[0])
    sub = tag(umin_jobs(cs=[P["c_min_sub"]])[0])
    sgap = C1.method_tag({"method": "S_gap", "k": 0, "steps": P["steps"], "seedset": P["seedset_main"]})
    jve = judge_rule(res_ve, main, srange=srange_ve)
    jvp = judge_rule(res_vp, main, srange=srange_vp)
    J = {"HC5": jve.get("a", jve["status"]), "HC6": jve.get("b", jve["status"]),
         "HC7": jvp.get("a", jvp["status"]), "HC8": jvp.get("b", jvp["status"]),
         "detail_VE_main": jve, "detail_VP_main": jvp,
         "obs_VE_sub": judge_rule(res_ve, sub, srange=srange_ve),
         "obs_VP_sub": judge_rule(res_vp, sub, srange=srange_vp),
         "obs_VE_S_gap": judge_rule(res_ve, sgap, srange=srange_ve),
         "obs_VP_S_gap_C1": judge_rule(res_vp, sgap, srange=srange_vp)}
    return J


# ---------------------------------------------------------------------------
# 確かめ：C2 の U_min の最初の評価（呼び出し 0）が、C1 の S_gap の呼び出し 0 と同じか
# （出発点・5% の点・網・D_G が同じなら、呼び出し 0 の u は同じ。呼び出し 1 からは、
#   C1 の S_gap が 5% の点を網の出力で進め、U_min（B21）は D_G で進めるので、比べない）
# ---------------------------------------------------------------------------
def replication_vs_c1(c1_out, c2_out, steps=None, seedset=None, c_min=None):
    steps = PREREG2["steps"] if steps is None else steps
    seedset = PREREG2["seedset_main"] if seedset is None else seedset
    c_min = PREREG2["c_min_main"] if c_min is None else c_min
    d1 = os.path.join(c1_out, "runs", C1.method_tag({"method": "S_gap", "k": 0, "steps": steps, "seedset": seedset}))
    d2 = os.path.join(c2_out, "runs", tag({"method": "U_min", "c_min": c_min, "steps": steps, "seedset": seedset}))
    rows = []
    worst = 0.0
    n_cmp = 0
    if not (os.path.isdir(d1) and os.path.isdir(d2)):
        return {"status": "比べるものがありません", "rows": rows}
    for f in sorted(x for x in os.listdir(d2) if x.endswith(".json")):
        p1 = os.path.join(d1, f)
        if not os.path.exists(p1):
            continue
        with open(p1, encoding="utf-8") as g:
            a = json.load(g)
        with open(os.path.join(d2, f), encoding="utf-8") as g:
            b = json.load(g)
        lim = min(a["switch_call"] if a.get("switch_call") is not None else 10 ** 9,
                  b["switch_call"] if b.get("switch_call") is not None else 10 ** 9)
        ta = {int(r[0]): r for r in a["trace"] if len(r) >= 4}
        tb = {int(r[0]): r for r in b["trace"]}
        cmp_calls = [c for c in sorted(tb) if c < lim and c in ta and c == 0]
        rel = [abs(tb[c][2] - ta[c][3]) / max(abs(ta[c][3]), 1e-300) for c in cmp_calls]
        rel_gap = [abs(tb[c][3] - ta[c][2]) / max(abs(ta[c][2]), 1e-300) for c in cmp_calls]
        w = max(rel + rel_gap) if cmp_calls else float("nan")
        if cmp_calls:
            worst = max(worst, w)
            n_cmp += len(cmp_calls)
        rows.append({"batch": f, "calls_compared": cmp_calls, "max_rel_diff": w})
    ok = n_cmp > 0 and worst <= PREREG2["replication_rel_tol"]
    return {"status": "再現" if ok else ("比べる呼び出しなし" if n_cmp == 0 else "ずれあり"),
            "max_rel_diff": worst, "n_compared": n_cmp, "rows": rows}


# ---------------------------------------------------------------------------
# 図
# ---------------------------------------------------------------------------
def make_figures2(out_dir, outs, res_by, judged, P=PREREG2):
    """outs = {"VP": c2 の vp フォルダ, "VE": ve フォルダ}、res_by = {"VP": res, "VE": res}"""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    os.makedirs(out_dir, exist_ok=True)
    files = []
    # 図 1：u の軌跡と切り替えの位置
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for ax, name in zip(axes, ("VP", "VE")):
        o = outs.get(name)
        drawn = False
        for c, col in ((P["c_min_main"], "C3"), (P["c_min_sub"], "C0")):
            d = os.path.join(o, "runs", tag({"method": "U_min", "c_min": c, "steps": P["steps"],
                                               "seedset": P["seedset_main"]})) if o else None
            if not d or not os.path.isdir(d):
                continue
            for i, fj in enumerate(sorted(x for x in os.listdir(d) if x.endswith(".json"))):
                with open(os.path.join(d, fj), encoding="utf-8") as f:
                    inf = json.load(f)
                tr = np.array([r[:3] for r in inf["trace"]], dtype=float)
                if len(tr):
                    ax.loglog(tr[:, 1], tr[:, 2], ".-", color=col, alpha=0.35,
                              label=("u traces, C_MIN = %g" % c) if i == 0 else None)
                    drawn = True
                if inf["switch_sigma"] > 0:
                    ax.axvline(inf["switch_sigma"], color=col, alpha=0.25, ls="--")
        jd = judged.get("detail_%s_main" % name, {})
        if jd.get("WV_tele_best"):
            ax.axvline(jd["WV_tele_best"]["sigma"], color="C2", lw=2, label="sigma of best WV_tele")
        ax.set_title("%s: u = median |D_G - D_net|^2 / d (5%% subset); dashed = switch" % name)
        ax.set_xlabel("sigma")
        ax.invert_xaxis()
        if drawn:
            ax.legend(fontsize=7)
    fig.tight_layout()
    p = os.path.join(out_dir, "fig1_u_traces_switch.png")
    fig.savefig(p, dpi=130)
    plt.close(fig)
    files.append(p)
    # 図 2：FID と費用
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for ax, name in zip(axes, ("VP", "VE")):
        res = res_by.get(name, {})
        g = sorted((r["cost_ratio_mean"], r["fid"]) for r in res.values()
                   if r["method"] == "WV_tele" and r["seedset"] == P["seedset_main"] and "fid" in r)
        if g:
            ax.plot([a for a, b in g], [b for a, b in g], "s-", color="gray", label="WV_tele(k)")
        for tg, r in res.items():
            if "fid" not in r or r["seedset"] != P["seedset_main"]:
                continue
            if r["method"] == "M0":
                ax.plot(r["cost_ratio_mean"], r["fid"], "ko", ms=9, label="M0")
            elif r["method"] == "S_gap":
                ax.plot(r["cost_ratio_mean"], r["fid"], "m^", ms=9, label="S_gap")
            elif r["method"] == "U_min":
                main = abs(r.get("c_min", 0) - P["c_min_main"]) < 1e-9
                ax.plot(r["cost_ratio_mean"], r["fid"], "r*" if main else "b*", ms=13,
                        label="U_min %g" % r.get("c_min", 0))
        jd = judged.get("detail_%s_main" % name, {})
        if jd.get("fid_tol"):
            ax.axhline(jd["fid_tol"], color="gray", ls=":", label="1.03 x FID(M0)")
        ax.axvline(P["cost_max"], color="gray", ls="--", lw=0.8)
        ax.set_xlabel("net evaluations / M0")
        ax.set_ylabel("FID (10k images)")
        ax.set_title(name)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize=7)
    fig.tight_layout()
    p = os.path.join(out_dir, "fig2_fid_vs_cost.png")
    fig.savefig(p, dpi=130)
    plt.close(fig)
    files.append(p)
    return files


def phase_report2(root, c1_out, ref, srange_vp=None, srange_ve=None, extra=None, n_images=None, batch=None,
                  log=print):
    vp = os.path.join(root, "vp")
    ve = os.path.join(root, "ve")
    kw = {"n_images": n_images or PREREG2["n_images"], "batch": batch or PREREG2["batch"]}
    res_vp = {}
    if os.path.isdir(c1_out):
        res_vp.update(C1.collect(c1_out, ref=ref, **kw))
    if os.path.isdir(vp):
        res_vp.update(collect2(vp, ref=ref, **kw))
    res_ve = collect2(ve, ref=ref, **kw) if os.path.isdir(ve) else {}
    J = judge_all(res_vp, res_ve, srange_vp, srange_ve)
    rep = os.path.join(root, "report")
    J["replication_VP_vs_C1"] = replication_vs_c1(c1_out, vp)
    figs = make_figures2(os.path.join(root, "figures"), {"VP": vp, "VE": ve}, {"VP": res_vp, "VE": res_ve}, J)
    summ = {"version": C2_VERSION, "prereg": PREREG2, "results_VP": res_vp, "results_VE": res_ve,
            "judgement": J, "figures": [os.path.basename(f) for f in figs], "extra": extra or {}}
    os.makedirs(rep, exist_ok=True)
    C1._atomic_json(os.path.join(rep, "summary_C2.json"), summ)
    for k in ("HC5", "HC6", "HC7", "HC8"):
        log("%s：%s" % (k, J[k]))
    log("VP の再現（C1 の S_gap の軌跡と）：%s" % J["replication_VP_vs_C1"]["status"])
    return summ


def estimate_minutes(cost_units_list, sec_per_m0_batch=232.0, n_batches=10, factor=1.0):
    """網の評価（M0 = 1 の単位）の合計から、時間（分）を見積もる。"""
    return float(sum(cost_units_list)) * sec_per_m0_batch * n_batches * factor / 60.0


def pack_report2(root, name="C2_report", max_files=100, top="C2_results"):
    return C1.pack_report(root, name=name, max_files=max_files, top=top)


In [ ]:
# ④ 読み込み
sys.path.insert(0, '/content')
import importlib, c1core as C1, c2core as C2
importlib.reload(C1); importlib.reload(C2)
print(C1.C1_VERSION, C2.C2_VERSION)
print(json.dumps(C2.PREREG2, ensure_ascii=False))


In [ ]:
# ⑤ s_G：CIFAR-10 の平均と共分散（C1 で作ったものを読む。なければ C1 と同じ手順で作る）
import torchvision
st_path = os.path.join(C1_OUT, 'cifar10_moments.npz')
if not os.path.exists(st_path):
    st_path = os.path.join(OUT, 'cifar10_moments.npz')
    if not os.path.exists(st_path):
        ds = torchvision.datasets.CIFAR10('/content/cifar', train=True, download=True)
        X = ds.data.transpose(0, 3, 1, 2).reshape(len(ds.data), -1)
        mu, cov, n = C1.moments_from_batches(X[i:i + 5000].astype(np.float64) / 127.5 - 1 for i in range(0, len(X), 5000))
        lam, U, n_neg = C1.eig_moments(cov)
        np.savez(st_path + '.tmp.npz', mu=mu, lam=lam, U=U, n=n, n_neg=n_neg)
        os.replace(st_path + '.tmp.npz', st_path)
z = np.load(st_path)
to_dev = lambda a: torch.as_tensor(a, dtype=torch.float64, device=dev)
G = C1.GaussianModel(z['mu'], z['lam'], z['U'], to_backend=to_dev)
print('積率：', st_path, ' 枚数', int(z['n']), ' 固有値の和', float(z['lam'].sum()))


In [ ]:
# ⑥ 網の読み込みと、出発点の乱数（C1 と同じ）。FID の道具
def load_net(url):
    with dnnlib.util.open_url(url) as f:
        net = pickle.load(f)['ema'].to(dev).eval()
    info = {'url': url, 'class': type(net).__name__, 'sigma_min': float(net.sigma_min),
            'sigma_max': float(net.sigma_max), 'sigma_data': float(getattr(net, 'sigma_data', float('nan')))}
    srange = C2.sampler_range(net.sigma_min, net.sigma_max)
    info['sampler_sigma_range'] = list(srange)
    # EDM の edm_sampler は net.round_sigma を通す。目盛りが変わらないことを確かめる
    sig = C2.sigmas_for(C2.PREREG2['steps'], srange)
    rs = net.round_sigma(torch.as_tensor(sig[:-1], dtype=torch.float64)).cpu().numpy()
    info['round_sigma_max_change'] = float(np.max(np.abs(rs - sig[:-1])))
    assert info['round_sigma_max_change'] < 1e-12, 'round_sigma が目盛りを変えます。止まって本筋に相談してください。'
    print(json.dumps(info, ensure_ascii=False))

    @torch.no_grad()
    def net_fn(x, sigma):
        out = torch.empty_like(x)
        s = torch.as_tensor(sigma, dtype=torch.float64, device=dev)
        for i in range(0, x.shape[0], NET_CHUNK):
            xb = x[i:i + NET_CHUNK].reshape(-1, 3, 32, 32)
            out[i:i + NET_CHUNK] = net(xb, s).to(torch.float64).reshape(xb.shape[0], -1)
        return out
    return net, net_fn, srange, info

def make_latents(seedset, b, n):   # C1 と同じ
    g = torch.Generator(device=dev).manual_seed(1000003 * int(seedset) + int(b))
    return torch.randn(n, 3072, generator=g, device=dev, dtype=torch.float64)

sync = torch.cuda.synchronize

with dnnlib.util.open_url(DETECTOR_URL) as f:
    detector = pickle.load(f).to(dev)
with dnnlib.util.open_url(REF_URL) as f:
    refz = dict(np.load(f))
ref = (refz['mu'], refz['sigma'])

@torch.no_grad()
def feat_fn(imgs_u8):
    fs = []
    for i in range(0, len(imgs_u8), 250):
        t = torch.as_tensor(imgs_u8[i:i + 250], device=dev)
        fs.append(detector(t, return_features=True).to(torch.float64).cpu().numpy())
    return np.concatenate(fs)

NET_INFO = {}


In [ ]:
# ⑦ VP の網：U_min（2.38、7.35）。M0・WV の格子・S_gap は C1 の結果を使う（走らせ直さない）
if RUN_VP:
    need = [C1.method_tag(j) for j in C1.main_jobs()]
    missing = [t for t in need if not os.path.exists(os.path.join(C1_OUT, 'runs', t, 'feats_10000.npy'))]
    print('C1 の結果で足りないもの：', missing if missing else 'なし')
    net, net_fn, srange_vp, NET_INFO['VP'] = load_net(VP_URL)
    assert abs(srange_vp[0] - 0.002) < 1e-12 and abs(srange_vp[1] - 80.0) < 1e-12, 'VP の目盛りが C1 と違います'
    vp_dir = os.path.join(OUT, 'vp')
    with torch.no_grad():
        C2.phase_runs2(vp_dir, C2.umin_jobs(), net_fn, G, make_latents, srange=srange_vp, sync=sync)
    C1.phase_features(vp_dir, [C2.tag(j) for j in C2.umin_jobs()], feat_fn)
    rep = C2.replication_vs_c1(C1_OUT, vp_dir)
    print('C1 の S_gap の軌跡との再現の確かめ：', rep['status'], ' 最大の相対差', rep.get('max_rel_diff'), ' 比べた呼び出し', rep.get('n_compared'))
    del net, net_fn; gc.collect(); torch.cuda.empty_cache()


In [ ]:
# ⑧ VE の網：規則に関わらない部分（M0、WV_tele の格子、別の種の M0）
if RUN_VE_BASE or RUN_VE_RULES:
    net, net_fn, srange_ve, NET_INFO['VE'] = load_net(VE_URL)
    ve_dir = os.path.join(OUT, 've')
    os.makedirs(ve_dir, exist_ok=True)
    json.dump(NET_INFO['VE'], open(os.path.join(ve_dir, 'net_info.json'), 'w'), ensure_ascii=False, indent=1)
    sig_ve = C2.sigmas_for(C2.PREREG2['steps'], srange_ve)
    print('VE の 18 歩の σ：', np.round(sig_ve, 4).tolist())
    print('WV_tele の k の格子：', C1.wv_k_grid(sig_ve, C2.PREREG2['wv_sigma_targets']))
if RUN_VE_BASE:
    base = C2.ve_base_jobs(sig_ve)
    t0 = time.time()
    with torch.no_grad():
        C2.phase_runs2(ve_dir, base[:1], net_fn, G, make_latents, srange=srange_ve, n_images=1000, sync=sync)
    j0 = json.load(open(os.path.join(ve_dir, 'runs', C2.tag(base[0]), 'b000.json')))
    sec = j0['seconds']
    units = [1.0, 1.0] + [(2 * (18 - j['k']) - 1) / 35 for j in base if j['method'] == 'WV_tele']
    print('VE の M0 1 束：%.0f 秒（VP は約 232 秒）。規則に関わらない部分の残りの見込み：約 %.1f 時間' %
          (sec, C2.estimate_minutes(units, sec_per_m0_batch=sec) / 60 - sec / 3600))
    with torch.no_grad():
        C2.phase_runs2(ve_dir, base, net_fn, G, make_latents, srange=srange_ve, sync=sync)
    C1.phase_features(ve_dir, [C2.tag(j) for j in base], feat_fn)


In [ ]:
# ⑨ VE の網：規則（U_min 2.38・7.35、S_gap）。S_gap の枝は C1 と同じパイロットで決める
#    （u の曲線は、ここでは表示しない）
if RUN_VE_RULES:
    with torch.no_grad():
        dec = C2.phase_pilot2(ve_dir, net_fn, G, make_latents, srange=srange_ve, log=lambda s: None)
    print('S_gap の枝：', dec['branch'])
    rules = C2.ve_rule_jobs()
    with torch.no_grad():
        C2.phase_runs2(ve_dir, rules, net_fn, G, make_latents, srange=srange_ve, branch=dec['branch'], sync=sync)
    C1.phase_features(ve_dir, [C2.tag(j) for j in rules], feat_fn)


In [ ]:
# ⑩ 集計・判定（HC5〜HC8、事前登録の基準）・図・zip
srange_ve_saved = None
p = os.path.join(OUT, 've', 'net_info.json')
if os.path.exists(p):
    srange_ve_saved = tuple(json.load(open(p))['sampler_sigma_range'])
extra = {'edm_commit': EDM_COMMIT, 'gpu': GPU_NAME, 'torch': torch.__version__, 'net_info': NET_INFO,
         'srange_ve': srange_ve_saved}
c1sum = os.path.join(C1_OUT, 'report', 'summary_C1.json')
summ = C2.phase_report2(OUT, C1_OUT, ref, srange_vp=None, srange_ve=srange_ve_saved, extra=extra)
# 確かめ：C1 の結果から計算し直した VP の FID が、summary_C1.json と同じか
if os.path.exists(c1sum):
    s1 = json.load(open(c1sum))['results']
    rv = summ['results_VP']
    d = max([abs(s1[t]['fid'] - rv[t]['fid']) for t in s1 if 'fid' in s1[t] and 'fid' in rv.get(t, {})] or [float('nan')])
    print('C1 の FID の計算し直しとの最大の差：', d)
print(json.dumps({k: summ['judgement'][k] for k in ('HC5', 'HC6', 'HC7', 'HC8')}, ensure_ascii=False, indent=1))
from IPython.display import Image, display
for f in summ['figures']:
    display(Image(os.path.join(OUT, 'figures', f)))
zips, nf = C2.pack_report2(OUT)
print('持ち帰る zip：', zips, '（%d ファイル）' % nf)
